In [0]:
%python

# Databricks notebook source
# MAGIC %md
# MAGIC # Export BI Summary Metrics
# MAGIC
# MAGIC Lives in `bi/`. Computes a small, cheap set of summary numbers independently of
# MAGIC `01_bi_analytics_dashboard.py` (rather than depending on that notebook's internal
# MAGIC variable names, which may have changed when it was rebuilt) and writes them to a
# MAGIC single-row Delta table.
# MAGIC
# MAGIC `ai/02_business_intelligence_assistant.py` reads this table -- it's the only thing
# MAGIC that connects the BI layer to the AI layer, so the two folders stay decoupled.
# MAGIC
# MAGIC Cheap by design: a handful of single-pass groupBys, not the full dashboard.

# COMMAND ----------

# Load data directly from table (replaces %run ./00_data_setup dependency)
from pyspark.sql.functions import sum as spark_sum, count as spark_count, col

TABLE_NAME = "miya_academy.default.whatsapp_messages"
SUMMARY_TABLE = "miya_academy.default.bi_summary_metrics"

# Load and prepare data
df_raw = spark.table(TABLE_NAME)
df_analysis = df_raw
df_filtered = df_analysis  # Both point to the same DataFrame in this context

# Calculate base metrics
msg_count = df_analysis.count()
convo_count = df_analysis.select("ConversationId").distinct().count()

print(f"Loaded {msg_count:,} messages from {convo_count:,} conversations")

# COMMAND ----------

# --- volume ---
vol_rows = {r["sender_type"]: r["total_messages"] for r in
            df_analysis.groupBy("sender_type").agg(spark_count("*").alias("total_messages")).collect()}
total_messages = msg_count
user_messages = vol_rows.get("User", 0)
bot_messages = vol_rows.get("Bot", 0)

# --- early drop-off (<=3 messages) ---
convo_lengths = df_filtered.groupBy("ConversationId").agg(spark_count("*").alias("total_messages"))
early_dropoff_count = convo_lengths.filter(col("total_messages") <= 3).count()
early_dropoff_pct = early_dropoff_count / convo_count * 100

# --- who sends the last message ---
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window
window_last = Window.partitionBy("ConversationId").orderBy(col("timestamp").desc())
last_messages = df_filtered.withColumn("msg_rank", row_number().over(window_last)).filter(col("msg_rank") == 1)
ending_rows = {r["sender_type"]: r["count"] for r in
               last_messages.groupBy("sender_type").agg(spark_count("*").alias("count")).collect()}
bot_last_pct = ending_rows.get("Bot", 0) / convo_count * 100

# --- top menu option ---
user_selections = df_filtered.filter(
    (col("sender_type") == "User") & (col("Text").rlike("^[0-9]{1,2}$"))
).groupBy("Text").agg(spark_count("*").alias("count")).orderBy(col("count").desc())
total_selections = user_selections.agg(spark_sum("count").alias("t")).collect()[0]["t"] or 1
top_selection_rows = user_selections.limit(2).collect()
top_menu_option = top_selection_rows[0]["Text"] if top_selection_rows else None
top_menu_option_pct = (top_selection_rows[0]["count"] / total_selections * 100) if top_selection_rows else 0
top2_menu_share = (sum(r["count"] for r in top_selection_rows) / total_selections * 100) if top_selection_rows else 0

# --- negation / frustration ---
negation_msgs = df_filtered.filter(
    (col("sender_type") == "User") &
    (col("Text").rlike("(?i)\\b(cancel|no|nope|stop)\\b"))
)
negation_count = negation_msgs.count()
negation_pct_of_user = (negation_count / user_messages * 100) if user_messages else 0

frustrated_conversations = negation_msgs.groupBy("ConversationId").agg(
    spark_count("*").alias("n")
).filter(col("n") > 1).count()

print("=" * 70)
print("BI SUMMARY METRICS")
print("=" * 70)
print(f"Total messages: {total_messages:,}")
print(f"Total conversations: {convo_count:,}")
print(f"Early drop-off (<=3 msg): {early_dropoff_count:,} ({early_dropoff_pct:.1f}%)")
print(f"Bot sent last message: {ending_rows.get('Bot', 0):,} ({bot_last_pct:.1f}%)")
print(f"Top menu option: {top_menu_option} ({top_menu_option_pct:.1f}%)")
print(f"Negations: {negation_count:,} ({negation_pct_of_user:.2f}% of user messages)")
print(f"Frustrated conversations (2+ negations): {frustrated_conversations:,}")

# COMMAND ----------

from pyspark.sql import Row

summary_row = Row(
    total_messages=total_messages,
    total_conversations=convo_count,
    avg_messages_per_conversation=float(total_messages / convo_count),
    user_message_pct=float(user_messages / total_messages * 100),
    bot_message_pct=float(bot_messages / total_messages * 100),
    early_dropoff_count=early_dropoff_count,
    early_dropoff_pct=float(early_dropoff_pct),
    bot_last_message_pct=float(bot_last_pct),
    top_menu_option=str(top_menu_option),
    top_menu_option_pct=float(top_menu_option_pct),
    top2_menu_share_pct=float(top2_menu_share),
    negation_count=negation_count,
    negation_pct_of_user_messages=float(negation_pct_of_user),
    frustrated_conversations=frustrated_conversations,
)

summary_df = spark.createDataFrame([summary_row])
summary_df.write.format("delta").mode("overwrite").saveAsTable(SUMMARY_TABLE)

print(f"\nWrote summary to {SUMMARY_TABLE}")
